Batch Correction Demo From Carson 03/08

Libraries

In [1]:
library(tidyverse)
library(BiocParallel)
library(parallel)
library(sva)

source("../rutils/R/dev_config.R")
source("../rutils/R/preprocessing.R")

── Attaching packages ───────────────────────────────────────────────────────────────────────────────────────────── tidyverse 1.3.1 ──

✔ ggplot2 3.3.5     ✔ purrr   0.3.4
✔ tibble  3.1.6     ✔ dplyr   1.0.8
✔ tidyr   1.2.0     ✔ stringr 1.4.0
✔ readr   2.1.2     ✔ forcats 0.5.1

── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()

Loading required package: mgcv

Loading required package: nlme


Attaching package: ‘nlme’


The following object is masked from ‘package:dplyr’:

    collapse


This is mgcv 1.8-39. For overview type 'help("mgcv-package")'.

Loading required package: genefilter


Attaching package: ‘genefilter’


The following object is masked from ‘package:readr’:

    spec




Load Data

In [2]:
dsets <- c("GSE4888", "GSE6364", "GSE7305","GSE51981")
batch <- as.list(setNames(seq(1,length(dsets)),dsets))

dirs <- get_dev_directories("dev_path.txt")
#lnames <- load("../proj/proj_vals.Rdata")

In [3]:
#matrisome_df <- load_matrisome_df(paste0(dirs$data_dir, "/matrisome/matrisome_hs_masterlist.tsv")) %>%
#    select(gene_symbol, division, category)
rma_dfs <- list()
clinical_dfs <- list()

for (ds in dsets) {
    rma_dfs[[ds]] <- read_tsv(paste0(dirs$data_dir, "/preprocessed_GEO_data/", ds, ".tsv"))
    clinical_dfs[[ds]] <- read_tsv(paste0(dirs$data_dir, "/preprocessed_GEO_clinical_data/", ds, ".tsv"))
}

# Handle making data uniform
clinical_dfs$GSE4888 <- clinical_dfs$GSE4888 %>%
    filter(condition == "normal") %>%
    rename(sample_name = geo_accession)
rma_dfs$GSE4888 <- rma_dfs$GSE4888 %>%
    select(one_of(c("symbol", clinical_dfs$GSE4888$sample_name)))

clinical_dfs$GSE6364 <- clinical_dfs$GSE6364 %>%
    rename(sample_name = geo_accession)

clinical_dfs$GSE51981 <- clinical_dfs$GSE51981 %>%
    filter(!(tissue == "unknown" | phase == "ambiguous_unknown" | condition == "unspecified_pathology")) %>%
    mutate(title = as.character(title)) %>%
    rename(sample_name = geo_accession)
rma_dfs$GSE51981 <- rma_dfs$GSE51981 %>%
    select(one_of(c("symbol", clinical_dfs$GSE51981$sample_name)))

all_clinical_df <- bind_rows(clinical_dfs$GSE4888, clinical_dfs$GSE6364, clinical_dfs$GSE51981) %>%
    rowwise() %>%
    mutate(batch = as.double(batch[[series]])) %>%
    ungroup() %>%
    mutate(
        condition = factor(condition, levels = c("normal", "endometriosis")),
        phase = factor(phase, levels = c("proliferative", "secretory_early", "secretory_mid", "secretory_late"))
    )

Rows: 21405 Columns: 28
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (1): symbol
dbl (27): GSM109814, GSM109815, GSM109816, GSM109817, GSM109818, GSM109819, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 27 Columns: 7
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (7): geo_accession, series, title, condition, endometriosis_stage, phase...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 21405 Columns: 38
── Column specification ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimite

In [4]:
# Only need the "symbol" column from the first matrix (since all have same genes)
all_rma_df <- bind_cols(rma_dfs$GSE4888, rma_dfs$GSE6364[-1], rma_dfs$GSE51981[-1])

In [5]:
#all_rma_df

In [6]:
#all_clinical_df

In [7]:
#all(colnames(all_rma_df)[-1] == all_clinical_df$sample_name)

Prep data

In [8]:
rma_expr <- as.matrix(all_rma_df[-1])
rownames(rma_expr) <- all_rma_df$symbol
model_m <- model.matrix(~ condition, data = all_clinical_df)
batch <- all_clinical_df$batch

# Batch Correction

Choose ref batch

In [9]:
ref_batch <- match("GSE4888", dsets)

In [10]:
ref_batch

[1] 1

THE MAGIC

In [11]:
bc_rma_expr <- ComBat(rma_expr, batch = batch, mod = model_m, ref.batch = ref_batch)

Using batch =1as a reference batch (this batch won't change)

Found3batches

Adjusting for1covariate(s) or covariate level(s)

Standardizing Data across genes

Fitting L/S model and finding priors

Finding parametric adjustments

Adjusting the Data




In [12]:
bc_rma_expr_df <- bc_rma_expr %>%
    as_tibble(rownames = "symbol")

In [13]:
bc_rma_expr_df

symbol,GSM109815,GSM109816,GSM109817,GSM109828,GSM109829,GSM109830,GSM109831,GSM150190.CEL.gz,GSM150191.CEL.gz,⋯,GSM1256786,GSM1256787,GSM1256788,GSM1256791,GSM1256793,GSM1256796,GSM1256797,GSM1256798,GSM1256799,GSM1256800
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,5.685566,6.309144,6.558803,5.676761,5.713494,5.651049,6.095096,6.734897,6.311051,⋯,5.807814,5.799219,6.019293,5.726395,5.767185,5.945817,5.736467,5.966595,5.868001,5.931074
A1BG-AS1,6.354643,7.025457,7.019929,6.820971,6.986295,7.019873,6.614798,7.162333,7.692468,⋯,6.700732,6.763380,6.717756,6.878098,6.691540,6.742212,6.825399,6.669743,6.579116,6.651995
A1CF,6.017202,6.806242,5.876322,6.075240,6.048896,6.123009,6.480756,6.343271,6.492032,⋯,6.245289,6.031816,5.796513,6.262573,6.067917,6.291459,5.939246,6.274130,6.093899,5.928298
A2M,10.705147,10.982733,11.381476,11.434625,11.319421,11.669190,11.679939,11.021458,10.674658,⋯,11.147112,11.202095,11.563820,11.578159,11.268653,11.595849,11.414081,11.245569,11.330142,11.243454
A2M-AS1,6.193356,6.496668,6.377511,6.072465,5.664001,6.158695,5.950787,6.176828,5.792398,⋯,6.171834,6.317519,6.506197,5.722691,6.196140,6.186685,6.062160,6.451924,6.235170,5.969264
A2ML1,5.936272,6.090145,5.661428,5.772800,5.649532,5.542943,6.006635,5.752719,6.293693,⋯,5.671488,5.636839,5.811220,5.782499,5.628093,5.671767,5.643707,5.561749,5.711164,5.600078
A2MP1,6.193356,6.532157,5.972719,6.051718,6.131620,6.140529,6.239424,6.046551,6.383858,⋯,6.038328,6.151578,6.124441,6.200450,6.225230,6.266043,5.956293,6.194836,6.257100,6.103579
A4GALT,7.399494,7.314846,7.260542,6.702430,7.204924,7.250284,7.084704,7.514506,7.745656,⋯,7.101046,6.920609,6.783543,7.082056,7.143511,7.054332,6.971775,6.935188,7.100574,7.065269
A4GNT,6.212988,6.594580,6.254428,6.024918,6.118253,6.123767,6.236670,6.186598,6.752777,⋯,6.290891,6.244271,6.052435,6.181848,6.126513,6.163117,6.266918,6.129302,6.278243,6.216228
